# QUANT Model — Precomputed Features, Min Leaf 10, Positive 5× Weight, Threshold 0.40

This notebook trains and evaluates an `ExtraTreesClassifier` using the **already saved QUANT-transformed features**.

It does **not** fit or run `QUANTTransformer` again.

**Fixed configuration**
- Classifier: `ExtraTreesClassifier`
- `min_samples_leaf`: `10`
- Class weighting: negative class = `1`, positive class = `5`
- Probability threshold: `0.40`
- Training data: `X_train_quant_5_features.npy`
- Test data: `X_test_quant_5_features.npy`


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Imports and configuration

In [ ]:
from pathlib import Path
import json
import time

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

print("scikit-learn version:", sklearn.__version__)

In [ ]:
# Folder containing the already transformed QUANT feature matrices.
DATA_DIR = Path(
    "/content/drive/MyDrive/solar_flare_forecasting/Data/quant_transformer_features"
)

# Results from this fixed model will be written here.
OUTPUT_DIR = DATA_DIR / "min_leaf_10_positive_5x_threshold_040_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed model configuration.
MIN_SAMPLES_LEAF = 10
POSITIVE_CLASS_WEIGHT = 5.0
CLASS_WEIGHT_NAME = "positive_5x"
PROBABILITY_THRESHOLD = 0.40
N_ESTIMATORS = 200
RANDOM_SEED = 42
PREDICT_BATCH_SIZE = 20_000

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("min_samples_leaf:", MIN_SAMPLES_LEAF)
print("Class weighting:", CLASS_WEIGHT_NAME)
print("Probability threshold:", PROBABILITY_THRESHOLD)

## 3. Load the precomputed QUANT features

The `.npy` feature matrices are memory-mapped so they do not have to be copied fully into RAM when loaded.

In [ ]:
def require_file(directory: Path, filename: str) -> Path:
    path = directory / filename
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


x_train_path = require_file(DATA_DIR, "X_train_quant_5_features.npy")
x_test_path = require_file(DATA_DIR, "X_test_quant_5_features.npy")
y_train_path = require_file(DATA_DIR, "y_train.npy")
y_test_path = require_file(DATA_DIR, "y_test.npy")

paths = {
    "X_train_quant": x_train_path,
    "X_test_quant": x_test_path,
    "y_train": y_train_path,
    "y_test": y_test_path,
}

for name, path in paths.items():
    print(f"{name:18s} {path}")

In [ ]:
# The QUANT transformation is NOT repeated here.
X_train_quant = np.load(x_train_path, mmap_mode="r")
X_test_quant = np.load(x_test_path, mmap_mode="r")
y_train = np.load(y_train_path)
y_test = np.load(y_test_path)

if X_train_quant.ndim != 2 or X_test_quant.ndim != 2:
    raise ValueError(
        "Expected two-dimensional QUANT feature matrices. "
        f"Received train={X_train_quant.shape}, test={X_test_quant.shape}."
    )

if X_train_quant.shape[0] != len(y_train):
    raise ValueError(
        f"Training alignment error: {X_train_quant.shape[0]} feature rows "
        f"but {len(y_train)} labels."
    )

if X_test_quant.shape[0] != len(y_test):
    raise ValueError(
        f"Test alignment error: {X_test_quant.shape[0]} feature rows "
        f"but {len(y_test)} labels."
    )

all_labels = np.unique(np.concatenate([y_train, y_test]))
if len(all_labels) != 2 or 1 not in all_labels:
    raise ValueError(
        "Expected binary labels containing positive class 1. "
        f"Found labels: {all_labels.tolist()}"
    )

POSITIVE_LABEL = 1
NEGATIVE_LABEL = all_labels[all_labels != POSITIVE_LABEL][0].item()
CLASS_WEIGHT = {
    NEGATIVE_LABEL: 1.0,
    POSITIVE_LABEL: POSITIVE_CLASS_WEIGHT,
}

print("X_train_quant:", X_train_quant.shape, X_train_quant.dtype)
print("X_test_quant: ", X_test_quant.shape, X_test_quant.dtype)
print("y_train:      ", y_train.shape, y_train.dtype)
print("y_test:       ", y_test.shape, y_test.dtype)
print("Negative label:", NEGATIVE_LABEL)
print("Positive label:", POSITIVE_LABEL)
print("Class-weight mapping:", CLASS_WEIGHT)

## 4. Inspect class distributions

In [ ]:
def class_distribution(y):
    labels, counts = np.unique(y, return_counts=True)
    result = pd.DataFrame({"class": labels, "count": counts})
    result["percent"] = 100.0 * result["count"] / len(y)
    return result


print("Training-set class distribution")
display(class_distribution(y_train))

print("Test-set class distribution")
display(class_distribution(y_test))

## 5. Train the classifier

The positive class receives five times the weight of the negative class, and every terminal tree leaf must contain at least 10 training samples.

In [ ]:
classifier = ExtraTreesClassifier(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    class_weight=CLASS_WEIGHT,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

start_time = time.time()
classifier.fit(X_train_quant, y_train)
training_seconds = time.time() - start_time

print(f"Training completed in {training_seconds / 60:.2f} minutes.")
print("Classifier classes:", classifier.classes_)
print("min_samples_leaf:", classifier.min_samples_leaf)
print("Class weights:", classifier.class_weight)

## 6. Predict test probabilities and apply the 0.40 threshold

Probabilities are generated in batches to limit temporary memory use.

In [ ]:
def positive_class_probabilities(
    model,
    X,
    positive_label=1,
    batch_size=20_000,
):
    classes = list(model.classes_)
    if positive_label not in classes:
        raise ValueError(
            f"Positive label {positive_label} is not present in model classes {classes}."
        )

    positive_column = classes.index(positive_label)
    probability_chunks = []

    for start in tqdm(
        range(0, len(X), batch_size),
        desc="Predicting test probabilities",
    ):
        stop = min(start + batch_size, len(X))
        probabilities = model.predict_proba(X[start:stop])[:, positive_column]
        probability_chunks.append(probabilities)

    return np.concatenate(probability_chunks)


test_probabilities = positive_class_probabilities(
    classifier,
    X_test_quant,
    positive_label=POSITIVE_LABEL,
    batch_size=PREDICT_BATCH_SIZE,
)

test_predictions = np.where(
    test_probabilities >= PROBABILITY_THRESHOLD,
    POSITIVE_LABEL,
    NEGATIVE_LABEL,
).astype(y_test.dtype)

print("Probability threshold:", PROBABILITY_THRESHOLD)
print("Predicted positive cases:", int((test_predictions == POSITIVE_LABEL).sum()))
print("Predicted negative cases:", int((test_predictions == NEGATIVE_LABEL).sum()))

## 7. Evaluate the fixed model

In [ ]:
def calculate_binary_metrics(y_true, y_pred, probabilities):
    true_positive_mask = np.asarray(y_true) == POSITIVE_LABEL
    predicted_positive_mask = np.asarray(y_pred) == POSITIVE_LABEL

    tn, fp, fn, tp = confusion_matrix(
        true_positive_mask,
        predicted_positive_mask,
        labels=[False, True],
    ).ravel()

    pod = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    far = fp / (tp + fp) if (tp + fp) else np.nan
    tss = pod - fpr if np.isfinite(pod) and np.isfinite(fpr) else np.nan

    hss_denominator = (
        (tp + fn) * (fn + tn)
        + (tp + fp) * (fp + tn)
    )
    hss = (
        2 * ((tp * tn) - (fp * fn)) / hss_denominator
        if hss_denominator
        else np.nan
    )

    observed_positives = tp + fn
    predicted_positives = tp + fp
    frequency_bias = (
        predicted_positives / observed_positives
        if observed_positives
        else np.nan
    )

    return {
        "min_samples_leaf": int(MIN_SAMPLES_LEAF),
        "class_weight_name": CLASS_WEIGHT_NAME,
        "negative_class_weight": float(CLASS_WEIGHT[NEGATIVE_LABEL]),
        "positive_class_weight": float(CLASS_WEIGHT[POSITIVE_LABEL]),
        "probability_threshold": float(PROBABILITY_THRESHOLD),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_positive": precision_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "recall_positive": recall_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "f1_positive": f1_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "POD_recall": pod,
        "FPR": fpr,
        "FAR": far,
        "frequency_bias": frequency_bias,
        "TSS": tss,
        "HSS": hss,
        "roc_auc": roc_auc_score(true_positive_mask, probabilities),
        "average_precision_pr_auc": average_precision_score(
            true_positive_mask,
            probabilities,
        ),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


test_metrics = calculate_binary_metrics(
    y_test,
    test_predictions,
    test_probabilities,
)

metrics_df = pd.DataFrame([test_metrics])
display(metrics_df)

print()
print("Classification report")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[NEGATIVE_LABEL, POSITIVE_LABEL],
        zero_division=0,
    )
)

confusion = confusion_matrix(
    y_test,
    test_predictions,
    labels=[NEGATIVE_LABEL, POSITIVE_LABEL],
)
confusion_df = pd.DataFrame(
    confusion,
    index=[f"true_{NEGATIVE_LABEL}", f"true_{POSITIVE_LABEL}"],
    columns=[f"pred_{NEGATIVE_LABEL}", f"pred_{POSITIVE_LABEL}"],
)

print("Confusion matrix")
display(confusion_df)

## 8. Save the model and results

In [ ]:
classifier_path = (
    OUTPUT_DIR / "extra_trees_min_leaf_10_positive_5x_threshold_040.joblib"
)
bundle_path = (
    OUTPUT_DIR / "quant_model_bundle_min_leaf_10_positive_5x_threshold_040.joblib"
)
metrics_csv_path = (
    OUTPUT_DIR / "test_metrics_min_leaf_10_positive_5x_threshold_040.csv"
)
metrics_json_path = (
    OUTPUT_DIR / "test_metrics_min_leaf_10_positive_5x_threshold_040.json"
)
predictions_path = (
    OUTPUT_DIR / "test_predictions_min_leaf_10_positive_5x_threshold_040.csv"
)

joblib.dump(classifier, classifier_path)

model_bundle = {
    "classifier": classifier,
    "probability_threshold": PROBABILITY_THRESHOLD,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "class_weight_name": CLASS_WEIGHT_NAME,
    "class_weight": CLASS_WEIGHT,
    "positive_label": POSITIVE_LABEL,
    "negative_label": NEGATIVE_LABEL,
    "input_type": "precomputed QUANT feature matrix",
    "quant_transformer_file": str(DATA_DIR / "quant_transformer_5_features.joblib"),
    "n_estimators": N_ESTIMATORS,
    "random_seed": RANDOM_SEED,
}
joblib.dump(model_bundle, bundle_path)

metrics_df.to_csv(metrics_csv_path, index=False)

metrics_to_save = {}
for key, value in test_metrics.items():
    if isinstance(value, np.integer):
        metrics_to_save[key] = int(value)
    elif isinstance(value, np.floating):
        metrics_to_save[key] = float(value)
    else:
        metrics_to_save[key] = value

metrics_to_save["training_seconds"] = float(training_seconds)
metrics_to_save["n_training_cases"] = int(len(y_train))
metrics_to_save["n_test_cases"] = int(len(y_test))
metrics_to_save["n_quant_features"] = int(X_train_quant.shape[1])

with open(metrics_json_path, "w") as file:
    json.dump(metrics_to_save, file, indent=2)

predictions_df = pd.DataFrame({
    "test_index": np.arange(len(y_test)),
    "y_true": y_test,
    "positive_probability": test_probabilities,
    "y_pred_threshold_040": test_predictions,
})
predictions_df.to_csv(predictions_path, index=False)

print("Saved classifier: ", classifier_path)
print("Saved model bundle:", bundle_path)
print("Saved metrics CSV:", metrics_csv_path)
print("Saved metrics JSON:", metrics_json_path)
print("Saved predictions:", predictions_path)

## Final result to report

Use the values in `test_metrics_min_leaf_10_positive_5x_threshold_040.csv` as the held-out test performance for this fixed configuration:

- **min_samples_leaf:** 10
- **Class weight:** positive class 5×, negative class 1×
- **Threshold:** 0.40
